In [0]:
select source_key, count(ride_id) from nyc_citi_bike_catalog.bronze.raw_trips_tbl
group by source_key;

In [0]:
select
  audit_logs.data_month,
  raw_trips.source_key,
  count(*) as row_count
from
  nyc_citi_bike_catalog.bronze.raw_trips_tbl as raw_trips
    full outer join nyc_citi_bike_catalog.audit.audit_events_tbl as audit_logs
      on raw_trips.source_key = audit_logs.source_key
where
  audit_logs.stage = 'BRONZE'
group by
  audit_logs.data_month,
  raw_trips.source_key
order by
  audit_logs.data_month;

In [0]:
-- MERGE INTO nyc_citi_bike_catalog.audit.audit_events_tbl AS target
-- USING (
--     SELECT source_key, zip_key, source_zip_name, source_url, data_month
--     FROM (
--         SELECT
--             source_key,
--             zip_key,
--             source_zip_name,
--             source_url,
--             data_month,
--             ROW_NUMBER() OVER (
--                 PARTITION BY source_key
--                 ORDER BY extracted_at DESC, file_id DESC
--             ) AS rn
--         FROM nyc_citi_bike_catalog.audit.audit_files_tbl
--         WHERE source_key IS NOT NULL
--     )
--     WHERE rn = 1
-- ) AS source
-- ON target.source_key = source.source_key
--    AND target.stage = 'BRONZE'
--    AND target.status = 'SUCCESS'
-- WHEN MATCHED AND (
--     target.zip_key IS NULL
--     OR target.source_zip_name IS NULL
--     OR target.source_url IS NULL
--     OR target.data_month IS NULL
-- )
-- THEN UPDATE SET
--     target.zip_key = COALESCE(target.zip_key, source.zip_key),
--     target.source_zip_name = COALESCE(target.source_zip_name, source.source_zip_name),
--     target.source_url = COALESCE(target.source_url, source.source_url),
--     target.data_month = COALESCE(target.data_month, source.data_month);